# AIaaS — MLPerf Inference Runner (Standard tier)

The **leaderboard-grade, accuracy-gated** tier. This notebook drives the official
**MLPerf Inference reference benchmark** — LoadGen + the reference app — for the
**vision classification & detection** workloads (resnet50, retinanet, mobilenet),
modeled on the `vision/classification_and_detection/GettingStarted.ipynb` tutorial
in the **`kurtvalcorza/inference`** fork.

Unlike the proxy notebooks, this measures what MLPerf measures:
- a **LoadGen scenario** (Offline / Server / SingleStream) — real traffic, not a
  batch sweep
- an **accuracy gate** (Top-1 for resnet50, mAP for retinanet)
- the official **`mlperf_log_summary.txt`** (QPS / latency percentiles / VALID)

### Honest scope
This is **heavy**: it clones + builds the fork (LoadGen C++ + the app), downloads a
model, and needs a dataset. The defaults run a **smoke test** (mobilenet on a
*fake* imagenet) so the pipeline works end-to-end in minutes; switch `MODEL` +
point `DATA_DIR` at the real **ImageNet** (resnet50) or **OpenImages** (retinanet)
for a credible run. For a fully reproducible, submission-grade path use the
**MLCFlow** automation (`mlcr run-mlperf-inference-app ...`) — see the notes at the
end. The actual runs belong in a session scoped to the `inference` fork; this
notebook is the portable bridge.

## 1. Clone the fork + build LoadGen and the app
References your `kurtvalcorza/inference` fork (the Standard-tier home).

In [ ]:
import os, subprocess, sys, time, json

REPO_URL = "https://github.com/kurtvalcorza/inference"
INF = os.path.abspath("inference-ref")
APP = os.path.join(INF, "vision", "classification_and_detection")

def sh(cmd, cwd=None, env=None):
    print("$", " ".join(cmd), f"(cwd={cwd})" if cwd else "")
    p = subprocess.run(cmd, cwd=cwd, env=env, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-1500:])
    if p.returncode != 0:
        print("STDERR:", p.stderr[-1500:])
    return p

if not os.path.isdir(INF):
    sh(["git", "clone", "--recurse-submodules", "--depth", "1", REPO_URL, INF])

# Build LoadGen (C++ + Python bindings) and the reference app.
sh([sys.executable, "-m", "pip", "install", "-q", "pybind11"])
_lg = sh([sys.executable, "setup.py", "develop"], cwd=os.path.join(INF, "loadgen"),
         env={**os.environ, "CFLAGS": "-std=c++14"})
if _lg.returncode != 0:
    raise RuntimeError("LoadGen build failed (see STDERR above) — fix build deps and re-run.")
_app = sh([sys.executable, "setup.py", "develop"], cwd=APP)
if _app.returncode != 0:
    raise RuntimeError("Reference app build failed (see STDERR above).")
import mlperf_loadgen   # hard import: stop here if the build didn't produce the module
print("mlperf_loadgen OK")


## 2. Backend dependencies
ONNX Runtime backend (works everywhere; use `onnxruntime-gpu` on a GPU). PyTorch /
TensorFlow backends also work — see the app README.

In [ ]:
import torch
USE_GPU = torch.cuda.is_available()
ort_pkg = "onnxruntime-gpu" if USE_GPU else "onnxruntime"
sh([sys.executable, "-m", "pip", "install", "-q", ort_pkg,
    "pycocotools", "opencv-python-headless", "numpy", "pillow", "pandas"])
print("backend:", ort_pkg, "| GPU:", USE_GPU)


## 3. Environment & hardware capture

In [ ]:
import platform, datetime

def smi(field):
    try:
        out = subprocess.run(["nvidia-smi", f"--query-gpu={field}",
                              "--format=csv,noheader,nounits"],
                             capture_output=True, text=True, timeout=10)
        return [x.strip() for x in out.stdout.strip().splitlines() if x.strip()]
    except Exception:
        return []

def detect_platform():
    if "COLAB_GPU" in os.environ or "COLAB_RELEASE_TAG" in os.environ: return "colab"
    if "KAGGLE_KERNEL_RUN_TYPE" in os.environ: return "kaggle"
    if os.path.exists("/opt/.sagemakerinternal"): return "sagemaker-studio-lab"
    return "local/on-prem"

ENV = {
    "platform": detect_platform(),
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "gpu_name": (torch.cuda.get_device_name(0) if USE_GPU else "cpu"),
    "vram_total_gb": (round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
                      if USE_GPU else None),
    "cuda": torch.version.cuda, "driver": (smi("driver_version") or ["?"])[0],
    "torch": torch.__version__, "python": platform.python_version(),
}
print(json.dumps(ENV, indent=2))


## 4. Configuration
Defaults = **smoke test** (mobilenet + fake imagenet). For credible runs, switch
`MODEL` and point `DATA_DIR` at the real dataset (see notes).

In [ ]:
# --- choose the benchmark ---
MODEL    = "mobilenet"          # smoke. Real: "resnet50" (ImageNet) | "retinanet" (OpenImages)
BACKEND  = "onnxruntime"
DEVICE   = "gpu" if USE_GPU else "cpu"
SCENARIO = "Offline"            # Offline | Server | SingleStream | MultiStream
ACCURACY = True                 # also run the accuracy pass (gate)

# Model download URLs (MLPerf reference models)
MODEL_URLS = {
    "mobilenet": "https://zenodo.org/record/3157894/files/mobilenet_v1_1.0_224.onnx",
    "resnet50":  "https://zenodo.org/record/4735647/files/resnet50_v1.onnx",
    "retinanet": "https://zenodo.org/record/6617879/files/resnext50_32x4d_fpn.onnx",
}

# Smoke run keeps it short; remove EXTRA_OPS for a full/valid run.
EXTRA_OPS = "--time 10 --max-latency 0.2" if MODEL == "mobilenet" else ""

# Dataset: 'fake' builds a tiny stand-in imagenet (smoke only). 'real' uses DATA_DIR.
DATASET_MODE = "fake" if MODEL == "mobilenet" else "real"
DATA_DIR = ""   # set to your ImageNet val / OpenImages dir for a real run

OUT_DIR = os.path.abspath("mlperf_results")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"MODEL={MODEL}, BACKEND={BACKEND}, DEVICE={DEVICE}, SCENARIO={SCENARIO}, "
      f"dataset={DATASET_MODE}")


## 5. Get the model and dataset

In [ ]:
# model
model_file = os.path.join(APP, os.path.basename(MODEL_URLS[MODEL]))
if not os.path.exists(model_file) or os.path.getsize(model_file) == 0:
    if os.path.exists(model_file):
        os.remove(model_file)   # drop a 0-byte file from a previous failed download
    r = sh(["wget", "-q", MODEL_URLS[MODEL], "-O", model_file])
    if r.returncode != 0 or not os.path.exists(model_file) or os.path.getsize(model_file) == 0:
        if os.path.exists(model_file):
            os.remove(model_file)
        raise RuntimeError(f"model download failed or empty: {MODEL_URLS[MODEL]}")
print("model:", model_file, os.path.getsize(model_file), "bytes")

# dataset
if DATASET_MODE == "fake":
    sh(["bash", "tools/make_fake_imagenet.sh"], cwd=APP)
    DATA_DIR = os.path.join(APP, "fake_imagenet")
else:
    assert DATA_DIR and os.path.isdir(DATA_DIR), (
        "Set DATA_DIR to a real dataset (ImageNet val for resnet50, "
        "OpenImages for retinanet). See the app README for download scripts.")
print("DATA_DIR:", DATA_DIR)


## 6. Run the MLPerf reference benchmark
`run_local.sh <backend> <model> <device> --scenario <S> [--accuracy]`, driven by
LoadGen. Official logs land in `mlperf_log_summary.txt` / `mlperf_log_detail.txt`.

In [ ]:
run_env = {**os.environ, "MODEL_DIR": APP, "DATA_DIR": DATA_DIR, "EXTRA_OPS": EXTRA_OPS}
import glob

def run_local(extra=()):
    cmd = ["bash", "run_local.sh", BACKEND, MODEL, DEVICE, "--scenario", SCENARIO, *extra]
    t0 = time.time()
    p = sh(cmd, cwd=APP, env=run_env)
    return p, round(time.time() - t0, 1)

def newest(pattern, root):
    cands = sorted(glob.glob(os.path.join(root, "**", pattern), recursive=True),
                   key=os.path.getmtime)
    return cands[-1] if cands else None

# 1) PERFORMANCE run — the mode that writes QPS / latency / "Result is VALID".
#    (--accuracy runs LoadGen in AccuracyOnly mode, which emits NO performance summary,
#     so it must be a separate run; running only --accuracy gives an all-None result.)
perf_proc, elapsed = run_local()
print(f"\nperformance run rc={perf_proc.returncode} in {elapsed}s")

# Capture the perf summary now, scoped to THIS run's output dir, before the accuracy
# run can overwrite mlperf_log_summary.txt (and so a stale model's summary isn't picked).
run_out = os.path.join(APP, "output", f"{MODEL}-{BACKEND}-{DEVICE}")
SUMMARY_TXT = newest("mlperf_log_summary.txt", run_out) or newest("mlperf_log_summary.txt", APP)
RESULTS_JSON = newest("results.json", run_out) or newest("results.json", os.path.join(APP, "output"))

# 2) ACCURACY run — separate mode; produces the accuracy log, not a perf summary.
ACC_LOG = None
if ACCURACY:
    acc_proc, acc_elapsed = run_local(["--accuracy"])
    print(f"accuracy run rc={acc_proc.returncode} in {acc_elapsed}s")
    ACC_LOG = newest("mlperf_log_accuracy.json", run_out) or newest("accuracy.txt", run_out)

print("perf summary:", SUMMARY_TXT)
print("results.json:", RESULTS_JSON)


## 7. Parse + normalize results

In [ ]:
import re

def parse_summary(path):
    if not path or not os.path.exists(path):
        return {}
    out = {}
    for line in open(path):
        m = re.match(r"\s*(.+?)\s*:\s*(.+?)\s*$", line)
        if m:
            k, v = m.group(1).strip(), m.group(2).strip()
            num = re.match(r"^[-\d.eE]+$", v)
            out[k] = float(v) if num else v
    return out

SUMMARY = parse_summary(SUMMARY_TXT)
RESULTS = {}
if RESULTS_JSON and os.path.exists(RESULTS_JSON):
    try:
        RESULTS = json.load(open(RESULTS_JSON))
    except Exception as e:
        print("results.json parse error:", e)

NORM = {
    "schema": "mlperf-inference/1.0",
    "env": ENV, "model": MODEL, "backend": BACKEND, "device": DEVICE,
    "scenario": SCENARIO, "accuracy_pass": ACCURACY,
    "dataset_mode": DATASET_MODE, "elapsed_s": elapsed,
    "accuracy_log": ACC_LOG,
    "loadgen_summary": SUMMARY, "app_results": RESULTS,
}
tag = (ENV["platform"] + "_" + ENV["gpu_name"] + "_" + MODEL + "_" + SCENARIO).replace(" ", "-").replace("/", "-")
out = os.path.join(OUT_DIR, f"mlperf_{tag}.json")
with open(out, "w") as f:
    json.dump(NORM, f, indent=2)
print("Saved:", out)

# headline fields (key names vary slightly by MLPerf version)
def pick(d, *names):
    for n in names:
        for k, v in d.items():
            if n.lower() in k.lower():
                return v
    return None
print("\n==== MLPERF SUMMARY ====")
print(f"{ENV['gpu_name']} | {MODEL} | {SCENARIO} | {BACKEND}")
print("Result valid :", pick(SUMMARY, "Result is", "Result:"))
print("QPS/throughput:", pick(SUMMARY, "Samples per second", "Scheduled samples per second",
                              "Completed samples per second", "QPS"))
print("Mean latency  :", pick(SUMMARY, "Mean latency"))
print("90th pct ns   :", pick(SUMMARY, "90.00 percentile", "90th percentile"))
print("Accuracy      :", pick(RESULTS, "accuracy", "mAP") if RESULTS else "(see accuracy log)")


## 8. Notes — real runs, datasets, and the MLCFlow path

- **Datasets:** resnet50 → **ImageNet** `val` (manual download/registration);
  retinanet → **OpenImages** (the app has download scripts under `tools/`). Set
  `DATASET_MODE="real"` and `DATA_DIR` accordingly, and drop `EXTRA_OPS` so the run
  meets MLPerf's minimum duration/queries for a **VALID** result.
- **Scenarios:** `Offline` (max throughput) and `Server` (Poisson arrivals under a
  p99 bound) are the datacenter scenarios; `SingleStream`/`MultiStream` are edge.
- **Submission-grade / reproducible:** prefer **MLCFlow** —
  `pip install mlc-scripts` then `mlcr run-mlperf-inference-app --model=retinanet
  --scenario=Offline --device=cuda ...` — which handles models, datasets, accuracy,
  and the submission tree. See `SubmissionExample.ipynb` in the fork.
- **LLM Standard tier:** the same MLPerf suite has `llama3.1-8b` and `whisper`
  (v5.1+) — run those in the fork to get the accuracy-gated LLM/ASR numbers that
  complement our Comparable-tier vLLM notebook.
- This notebook is the portable bridge; the heavy/credible runs belong in an
  `inference`-scoped session.

## 9. The MLPerf Inference **suite** via MLCFlow

The vision flow above uses the fork's `run_local.sh`. For the rest of the MLPerf
Inference **suite**, use **MLCFlow / CM** — it fetches each model + dataset, runs
LoadGen, and applies the accuracy gate. This section takes a **list of models** and
runs them in a loop, so you get the suite in one pass.

**Datacenter suite** (set `MLC_MODELS` to the subset you want):
`resnet50`, `retinanet`, `3d-unet-99(.9)`, `bert-99(.9)`, `dlrm-v2-99(.9)`,
`gptj-99(.9)`, `sdxl`, `llama2-70b-99(.9)`, `mixtral-8x7b`, `llama3.1-8b`,
`whisper`, `deepseek-r1`. The default list is a **40 GB-fittable subset**; large
models (llama2-70b, mixtral, deepseek-r1) need more VRAM / multi-GPU.
`execution_mode=test` is a smoke run; use `valid` for submission-grade.

In [ ]:
import subprocess, sys, os, json

# CM automation (MLCFlow's predecessor interface; `mlc`/`mlcr` also work on newer installs).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cm4mlops"], check=False)
subprocess.run(["cm", "pull", "repo", "mlcommons@cm4mlops"], check=False)
import shutil
print("cm on PATH:", shutil.which("cm"), "| cmr:", shutil.which("cmr"))


In [ ]:
# MLPerf Inference SUITE via MLCFlow — set the models; the runner loops over them.
MLC_MODELS     = ["resnet50", "retinanet", "sdxl", "whisper"]   # 40 GB-fittable subset; edit freely
MLC_SCENARIO   = "Offline"
MLC_DEVICE     = "cuda" if USE_GPU else "cpu"
MLC_IMPL       = "mlcommons-python"     # reference implementation
MLC_EXEC_MODE  = "test"                 # "test" (smoke) | "valid" (submission-grade)
MLC_CATEGORY   = "datacenter"           # "datacenter" | "edge"
MLC_RESULTS    = os.path.abspath("mlperf_mlc_results")
os.makedirs(MLC_RESULTS, exist_ok=True)

import time
SUITE = {}
for model in MLC_MODELS:
    cmd = [
        "cmr", "run-mlperf,inference",
        f"--model={model}", f"--scenario={MLC_SCENARIO}", f"--device={MLC_DEVICE}",
        f"--implementation={MLC_IMPL}", f"--execution_mode={MLC_EXEC_MODE}",
        f"--category={MLC_CATEGORY}", f"--results_dir={MLC_RESULTS}", "--quiet=True",
    ]
    print("\n=== MLPerf:", model, "===\n$", " ".join(cmd))
    t0 = time.time()
    p = subprocess.run(cmd, capture_output=True, text=True)
    print(p.stdout[-1200:])
    if p.returncode != 0:
        print("STDERR:", p.stderr[-1200:])
    SUITE[model] = {"returncode": p.returncode, "elapsed_s": round(time.time() - t0, 1)}


In [ ]:
import glob, re
import pandas as pd

def parse_summary(path):
    out = {}
    if path and os.path.exists(path):
        for line in open(path):
            m = re.match(r"\s*(.+?)\s*:\s*(.+?)\s*$", line)
            if m:
                k, v = m.group(1).strip(), m.group(2).strip()
                out[k] = float(v) if re.match(r"^[-\d.eE]+$", v) else v
    return out

rows = []
for model in MLC_MODELS:
    # best-effort: pick the most recent summary that sits under a dir naming this model
    cands = [f for f in glob.glob(os.path.join(MLC_RESULTS, "**", "mlperf_log_summary.txt"), recursive=True)
             if model.split("-")[0] in f.lower()]
    cands = sorted(cands or glob.glob(os.path.join(MLC_RESULTS, "**", "mlperf_log_summary.txt"), recursive=True),
                   key=os.path.getmtime)
    summ = parse_summary(cands[-1]) if cands else {}
    NORM = {
        "schema": "mlperf-inference/1.0", "via": "mlcflow", "env": ENV, "model": model,
        "scenario": MLC_SCENARIO, "device": MLC_DEVICE, "implementation": MLC_IMPL,
        "execution_mode": MLC_EXEC_MODE, "category": MLC_CATEGORY,
        "returncode": SUITE[model]["returncode"], "elapsed_s": SUITE[model]["elapsed_s"],
        "loadgen_summary": summ,
    }
    with open(os.path.join(OUT_DIR, f"mlperf_mlc_{model}_{MLC_SCENARIO}.json"), "w") as f:
        json.dump(NORM, f, indent=2)
    rows.append({"model": model, "rc": SUITE[model]["returncode"],
                 "valid": summ.get("Result is", summ.get("Result")),
                 "qps": summ.get("Samples per second", summ.get("Scheduled samples per second")),
                 "elapsed_s": SUITE[model]["elapsed_s"]})

print("\n==== MLPERF INFERENCE SUITE ====")
df = pd.DataFrame(rows)
try:
    from IPython.display import display; display(df)
except Exception:
    print(df.to_string(index=False))
print("test mode = smoke; execution_mode=valid for VALID numbers. Per-model JSON in", OUT_DIR)
